In [1]:
import math
import re

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

print("Libraries imported successfully! ✅")


d:\college_work\PG\linkedIn_projects\TravelMate-AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully! ✅


In [2]:
DATA_PATH = "../data/processed/multi_city_places.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nCities:")
print(df["city"].value_counts())


Dataset shape: (120, 11)

Cities:
city
Manali       20
Goa          20
Jaipur       20
Udaipur      20
Rishikesh    20
Shimla       20
Name: count, dtype: int64


In [3]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate place IDs:")
print(df["place_id"].duplicated().sum())

print("\nRating range:")
print(df["rating"].min(), "to", df["rating"].max())


Missing values:
place_id        0
name            0
city            0
address         0
country         0
latitude        0
longitude       0
rating          0
reviews         0
category        0
source_query    0
dtype: int64

Duplicate place IDs:
0

Rating range:
3.6 to 4.9


In [4]:
text_columns = [
    "name",
    "city",
    "address",
    "country",
    "category"
]

for col in text_columns:
    df[col] = (
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

df["name_clean"] = (
    df["name"]
    .str.lower()
)

df["category_clean"] = (
    df["category"]
    .str.lower()
)

df[[
    "name",
    "city",
    "category"
]].head(10)


,name,city,category
0,Hadimba Devi Temple,Manali,Tourist attraction
1,Museum of Himachal Culture & Folk Art,Manali,Tourist attraction
2,Jogini Falls,Manali,Tourist attraction
3,Old Manali snow point,Manali,Tourist attraction
4,Nehru Kund,Manali,Tourist attraction
5,Mini Switzerland Manali,Manali,Tourist attraction
6,Baror Parsha Waterfall,Manali,Tourist attraction
7,Kullu Manali River rafting,Manali,Tourist attraction
8,Van Vihar National Park,Manali,Tourist attraction
9,Manali Bazaar,Manali,Tourist attraction


In [5]:
def make_feature_text(row):
    return (
        f"{row['name_clean']} "
        f"{row['category_clean']}"
    ).lower()


df["feature_text"] = df.apply(
    make_feature_text,
    axis=1
)


def has_any(text, keywords):
    return int(
        any(
            keyword in text
            for keyword in keywords
        )
    )


df["nature"] = df["feature_text"].apply(
    lambda x: has_any(
        x,
        [
            "beach", "waterfall", "lake",
            "park", "forest", "valley",
            "hill", "mountain", "garden",
            "nature", "falls", "river"
        ]
    )
)

df["history"] = df["feature_text"].apply(
    lambda x: has_any(
        x,
        [
            "fort", "palace", "museum",
            "heritage", "historical",
            "monument", "temple", "church",
            "mosque", "haveli", "city palace"
        ]
    )
)

df["culture"] = df["feature_text"].apply(
    lambda x: has_any(
        x,
        [
            "temple", "museum", "palace",
            "market", "bazaar", "church",
            "mosque", "heritage", "fort"
        ]
    )
)

df["adventure"] = df["feature_text"].apply(
    lambda x: has_any(
        x,
        [
            "trek", "rafting", "camp",
            "adventure", "sports", "ski",
            "snow", "water sports"
        ]
    )
)

df["photography"] = df["feature_text"].apply(
    lambda x: has_any(
        x,
        [
            "viewpoint", "view point",
            "waterfall", "beach", "lake",
            "fort", "palace", "sunset",
            "garden", "scenic"
        ]
    )
)

df["shopping"] = df["feature_text"].apply(
    lambda x: has_any(
        x,
        [
            "market", "bazaar",
            "shopping", "mall"
        ]
    )
)

df["religious"] = df["feature_text"].apply(
    lambda x: has_any(
        x,
        [
            "temple", "church",
            "mosque", "gurudwara",
            "monastery"
        ]
    )
)

df["family"] = df["feature_text"].apply(
    lambda x: has_any(
        x,
        [
            "park", "garden", "zoo",
            "museum", "beach",
            "aquarium", "family"
        ]
    )
)

feature_columns = [
    "nature",
    "history",
    "culture",
    "adventure",
    "photography",
    "shopping",
    "religious",
    "family"
]

df[[
    "name",
    "city",
    *feature_columns
]].head(20)


,name,city,nature,history,culture,adventure,photography,shopping,religious,family
0,Hadimba Devi Temple,Manali,0,1,1,0,0,0,1,0
1,Museum of Himachal Culture & Folk Art,Manali,0,1,1,0,0,0,0,1
2,Jogini Falls,Manali,1,0,0,0,0,0,0,0
3,Old Manali snow point,Manali,0,0,0,1,0,0,0,0
4,Nehru Kund,Manali,0,0,0,0,0,0,0,0
5,Mini Switzerland Manali,Manali,0,0,0,0,0,0,0,0
6,Baror Parsha Waterfall,Manali,1,0,0,0,1,0,0,0
7,Kullu Manali River rafting,Manali,1,0,0,1,0,0,0,0
8,Van Vihar National Park,Manali,1,0,0,0,0,0,0,1
9,Manali Bazaar,Manali,0,0,1,0,0,1,0,0


In [6]:
def build_tags(row):
    tags = [
        feature
        for feature in feature_columns
        if row[feature] == 1
    ]

    return ", ".join(tags)


df["travel_tags"] = df.apply(
    build_tags,
    axis=1
)

df[[
    "name",
    "city",
    "travel_tags"
]].head(20)


,name,city,travel_tags
0,Hadimba Devi Temple,Manali,"history, culture, religious"
1,Museum of Himachal Culture & Folk Art,Manali,"history, culture, family"
2,Jogini Falls,Manali,nature
3,Old Manali snow point,Manali,adventure
4,Nehru Kund,Manali,
5,Mini Switzerland Manali,Manali,
6,Baror Parsha Waterfall,Manali,"nature, photography"
7,Kullu Manali River rafting,Manali,"nature, adventure"
8,Van Vihar National Park,Manali,"nature, family"
9,Manali Bazaar,Manali,"culture, shopping"


In [7]:
def estimate_visit_minutes(row):
    text = row["feature_text"]

    if any(x in text for x in ["fort", "palace", "museum"]):
        return 120

    if any(x in text for x in ["waterfall", "falls", "beach", "lake"]):
        return 90

    if any(x in text for x in ["trek", "rafting", "adventure", "ski"]):
        return 150

    if any(x in text for x in ["temple", "church", "mosque"]):
        return 60

    if any(x in text for x in ["market", "bazaar", "mall"]):
        return 90

    if any(x in text for x in ["viewpoint", "view point"]):
        return 45

    return 60


df["estimated_visit_minutes"] = df.apply(
    estimate_visit_minutes,
    axis=1
)

df[[
    "name",
    "city",
    "estimated_visit_minutes"
]].head(10)


,name,city,estimated_visit_minutes
0,Hadimba Devi Temple,Manali,60
1,Museum of Himachal Culture & Folk Art,Manali,120
2,Jogini Falls,Manali,90
3,Old Manali snow point,Manali,60
4,Nehru Kund,Manali,60
5,Mini Switzerland Manali,Manali,60
6,Baror Parsha Waterfall,Manali,90
7,Kullu Manali River rafting,Manali,150
8,Van Vihar National Park,Manali,60
9,Manali Bazaar,Manali,90


In [8]:
df["recommendation_text"] = (
    df["name"] + ". "
    + df["city"] + ". "
    + df["category"] + ". "
    + df["travel_tags"]
)

df["recommendation_text"] = (
    df["recommendation_text"]
    .str.lower()
)

df[[
    "name",
    "recommendation_text"
]].head(10)


,name,recommendation_text
0,Hadimba Devi Temple,hadimba devi temple. manali. tourist attractio...
1,Museum of Himachal Culture & Folk Art,museum of himachal culture & folk art. manali....
2,Jogini Falls,jogini falls. manali. tourist attraction. nature
3,Old Manali snow point,old manali snow point. manali. tourist attract...
4,Nehru Kund,nehru kund. manali. tourist attraction.
5,Mini Switzerland Manali,mini switzerland manali. manali. tourist attra...
6,Baror Parsha Waterfall,baror parsha waterfall. manali. tourist attrac...
7,Kullu Manali River rafting,kullu manali river rafting. manali. tourist at...
8,Van Vihar National Park,van vihar national park. manali. tourist attra...
9,Manali Bazaar,manali bazaar. manali. tourist attraction. cul...


In [9]:
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    df["recommendation_text"]
)

print("TF-IDF matrix:", tfidf_matrix.shape)


TF-IDF matrix: (120, 551)


In [10]:
semantic_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

place_embeddings = semantic_model.encode(
    df["recommendation_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding matrix:", place_embeddings.shape)


Batches: 100%|██████████| 4/4 [00:00<00:00,  7.63it/s]

Embedding matrix: (120, 384)


In [11]:
def minmax(series):
    series = series.astype(float)

    if series.max() == series.min():
        return pd.Series(
            np.ones(len(series)),
            index=series.index
        )

    return (
        (series - series.min())
        / (series.max() - series.min())
    )


df["rating_score"] = minmax(df["rating"])

df["popularity_score"] = minmax(
    np.log1p(
        df["reviews"].clip(lower=0)
    )
)


In [12]:
def recommend_city(
    destination,
    query,
    user_preferences,
    top_n=5
):
    destination_clean = (
        destination.strip().lower()
    )

    city_mask = (
        df["city"].str.lower()
        == destination_clean
    )

    candidates = df[city_mask].copy()

    if candidates.empty:
        available_cities = sorted(
            df["city"].unique().tolist()
        )

        raise ValueError(
            f"Destination '{destination}' not found. "
            f"Available destinations: {available_cities}"
        )

    candidate_indices = candidates.index.to_numpy()

    # -----------------------------
    # Structured similarity
    # -----------------------------
    missing = [
        feature
        for feature in feature_columns
        if feature not in user_preferences
    ]

    if missing:
        raise ValueError(
            f"Missing preference features: {missing}"
        )

    user_vector = np.array([
        float(user_preferences[feature])
        for feature in feature_columns
    ]).reshape(1, -1)

    structured_matrix = (
        df.loc[
            candidate_indices,
            feature_columns
        ]
        .fillna(0)
        .astype(float)
    )

    structured_scores = cosine_similarity(
        user_vector,
        structured_matrix
    ).flatten()

    # -----------------------------
    # TF-IDF similarity
    # -----------------------------
    query_text = (
        f"{destination}. {query}"
    )

    query_tfidf = tfidf_vectorizer.transform(
        [query_text.lower()]
    )

    city_tfidf = tfidf_matrix[
        candidate_indices
    ]

    tfidf_scores = cosine_similarity(
        query_tfidf,
        city_tfidf
    ).flatten()

    # -----------------------------
    # Semantic similarity
    # -----------------------------
    query_embedding = semantic_model.encode(
        [query_text],
        normalize_embeddings=True
    )

    city_embeddings = place_embeddings[
        candidate_indices
    ]

    semantic_scores = cosine_similarity(
        query_embedding,
        city_embeddings
    ).flatten()

    # -----------------------------
    # Ranking
    # -----------------------------
    candidates["structured_score"] = structured_scores
    candidates["tfidf_score"] = tfidf_scores
    candidates["semantic_score"] = semantic_scores

    candidates["final_score"] = (
        0.25 * candidates["structured_score"]
        + 0.25 * candidates["tfidf_score"]
        + 0.30 * candidates["semantic_score"]
        + 0.10 * candidates["rating_score"]
        + 0.10 * candidates["popularity_score"]
    )

    return (
        candidates
        .sort_values(
            "final_score",
            ascending=False
        )
        [[
            "name",
            "city",
            "category",
            "rating",
            "reviews",
            "travel_tags",
            "estimated_visit_minutes",
            "structured_score",
            "tfidf_score",
            "semantic_score",
            "final_score"
        ]]
        .head(top_n)
        .reset_index(drop=True)
    )


In [13]:
preferences_nature_photo = {
    "nature": 1.0,
    "history": 0.0,
    "culture": 0.0,
    "adventure": 0.3,
    "photography": 1.0,
    "shopping": 0.0,
    "religious": 0.0,
    "family": 0.2
}

recommend_city(
    "Manali",
    "peaceful scenic places for photography",
    preferences_nature_photo,
    top_n=5
)


,name,city,category,rating,reviews,travel_tags,estimated_visit_minutes,structured_score,tfidf_score,semantic_score,final_score
0,Sajla Waterfall,Manali,Tourist attraction,4.7,1091,"nature, photography",90,0.969003,0.307788,0.770708,0.681269
1,Rahala Waterfalls,Manali,Tourist attraction,4.5,797,"nature, photography",90,0.969003,0.291573,0.733850,0.647444
2,Baror Parsha Waterfall,Manali,Tourist attraction,4.7,489,"nature, photography",90,0.969003,0.259131,0.708478,0.641930
3,Manali View Point,Manali,Tourist attraction,4.6,87,photography,45,0.685189,0.452924,0.839044,0.632677
4,Jogini Falls,Manali,Tourist attraction,4.6,10844,nature,90,0.685189,0.193740,0.626294,0.555154


In [14]:
recommend_city(
    "Goa",
    "beautiful beaches and relaxed scenic places",
    {
        "nature": 1.0,
        "history": 0.1,
        "culture": 0.4,
        "adventure": 0.4,
        "photography": 1.0,
        "shopping": 0.2,
        "religious": 0.0,
        "family": 0.5
    },
    top_n=5
)


,name,city,category,rating,reviews,travel_tags,estimated_visit_minutes,structured_score,tfidf_score,semantic_score,final_score
0,"Calangute Beach, Goa",Goa,Tourist attraction,4.3,103884,"nature, photography, family",90,0.891720,0.392560,0.830518,0.718666
1,Keri Beach,Goa,Tourist attraction,4.6,5070,"nature, photography, family",90,0.891720,0.221955,0.816173,0.662736
2,Kuske Waterfall,Goa,Tourist attraction,4.4,332,"nature, photography",90,0.873704,0.243288,0.731620,0.593910
3,Dudhsagar Falls,Goa,Tourist attraction,4.6,32335,nature,90,0.617802,0.244277,0.699319,0.584445
4,Cabo de Rama Fort South Goa,Goa,Tourist attraction,4.4,17073,"history, culture, photography",120,0.535032,0.315029,0.729781,0.568416


In [15]:
recommend_city(
    "Jaipur",
    "historical forts palaces and cultural attractions",
    {
        "nature": 0.1,
        "history": 1.0,
        "culture": 1.0,
        "adventure": 0.0,
        "photography": 0.8,
        "shopping": 0.4,
        "religious": 0.2,
        "family": 0.3
    },
    top_n=5
)


,name,city,category,rating,reviews,travel_tags,estimated_visit_minutes,structured_score,tfidf_score,semantic_score,final_score
0,Amber Palace,Jaipur,Tourist attraction,4.6,172870,"history, culture, photography",120,0.942809,0.067163,0.762339,0.658118
1,Nahargarh Fort,Jaipur,Castle,4.5,78755,"history, culture, photography",120,0.942809,0.060685,0.819977,0.657753
2,The City Palace,Jaipur,Tourist attraction,4.4,59335,"history, culture, photography",120,0.942809,0.068708,0.842417,0.655793
3,Jaigarh Fort,Jaipur,Tourist attraction,4.5,32119,"history, culture, photography",120,0.942809,0.068538,0.817280,0.649387
4,"Sheesh Mahal, Amber Fort",Jaipur,Tourist attraction,4.6,6301,"history, culture, photography",120,0.942809,0.053627,0.789681,0.627785


In [16]:
available_cities = sorted(
    df["city"].unique()
)

verification_rows = []

for city in available_cities:

    result = recommend_city(
        city,
        "places to visit and great travel experiences",
        {
            "nature": 0.5,
            "history": 0.5,
            "culture": 0.5,
            "adventure": 0.5,
            "photography": 0.5,
            "shopping": 0.2,
            "religious": 0.2,
            "family": 0.5
        },
        top_n=5
    )

    verification_rows.append({
        "requested_city": city,
        "returned_rows": len(result),
        "all_city_values_match":
            result["city"].eq(city).all()
    })

verification_df = pd.DataFrame(
    verification_rows
)

verification_df


,requested_city,returned_rows,all_city_values_match
0,Goa,5,True
1,Jaipur,5,True
2,Manali,5,True
3,Rishikesh,5,True
4,Shimla,5,True
5,Udaipur,5,True


In [17]:
city_reports = {}

default_preferences = {
    "nature": 0.7,
    "history": 0.3,
    "culture": 0.4,
    "adventure": 0.4,
    "photography": 0.7,
    "shopping": 0.3,
    "religious": 0.2,
    "family": 0.4
}

for city in available_cities:

    city_reports[city] = recommend_city(
        city,
        "best places for a memorable trip",
        default_preferences,
        top_n=5
    )

for city, report in city_reports.items():
    print(f"\n🌍 {city}")
    display(
        report[[
            "name",
            "rating",
            "reviews",
            "final_score"
        ]]
    )



🌍 Goa


,name,rating,reviews,final_score
0,"Calangute Beach, Goa",4.3,103884,0.626456
1,Keri Beach,4.6,5070,0.598228
2,Cabo de Rama Fort South Goa,4.4,17073,0.554375
3,Sinquerim Fort,4.5,20206,0.552308
4,Fort Aguada,4.2,111434,0.543716



🌍 Jaipur


,name,rating,reviews,final_score
0,Jai Niwas Garden,4.4,26478,0.598283
1,Amber Palace,4.6,172870,0.580046
2,The City Palace,4.4,59335,0.570613
3,Jaigarh Fort,4.5,32119,0.561158
4,Central Park,4.6,25012,0.554950



🌍 Manali


...

,name,rating,reviews,final_score
0,Goa Beach Rishikesh,4.5,2359,0.575508
1,Rishikesh family beach,4.5,97,0.559856
2,The Secret Waterfall Rishikesh - DhaulSrot ( B...,4.5,2815,0.553718
3,Rishikesh River Rafting,4.5,2063,0.552612
4,Rishikesh View Point,4.6,864,0.530869



🌍 Shimla


,name,rating,reviews,final_score
0,"Christ Church, Shimla",4.6,22716,0.541037
1,Marina Waterfall,4.2,344,0.515568
2,Shimla Hills,4.3,83,0.508128
3,View point shimla,4.7,83,0.507068
4,Rani Jhansi Park,4.3,1362,0.475405



🌍 Udaipur


,name,rating,reviews,final_score
0,"Pichola Lake, Udaipur",4.7,3076,0.600720
1,Sunset Point Park Udaipur,4.6,2246,0.589881
2,Monsoon Palace,4.4,34718,0.551540
3,City Palace,4.5,106511,0.550873
4,Wax Museum Udaipur,4.7,47100,0.537018


In [18]:
output_path = (
    "../data/processed/"
    "multi_city_recommendation_features.csv"
)

df.to_csv(
    output_path,
    index=False
)

print(f"✅ Saved: {output_path}")
print("Final shape:", df.shape)


✅ Saved: ../data/processed/multi_city_recommendation_features.csv
Final shape: (120, 27)
